# Поиск изображений по текстовому описанию — PoC

**With Sense** — демонстрация сопоставления текста и изображения.


In [ ]:
# %pip install -r ../requirements.txt


In [ ]:

import warnings
warnings.filterwarnings('ignore')

import re
import pickle
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from PIL import Image
from torchvision import models, transforms
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm


def find_repo_root() -> Path:
    markers = ('data/train_dataset.csv', 'train_dataset.csv')
    for root in [Path.cwd(), *Path.cwd().parents]:
        for marker in markers:
            if (root / marker).exists():
                return root.resolve()
    raise FileNotFoundError('Корень репозитория не найден (нет train_dataset.csv)')

REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / 'data' if (REPO_ROOT / 'data' / 'train_dataset.csv').exists() else REPO_ROOT
TRAIN_IMAGES_DIR = DATA_DIR / 'train_images'
TEST_IMAGES_DIR = DATA_DIR / 'test_images'
CACHE_DIR = REPO_ROOT / 'cache'
CACHE_DIR.mkdir(exist_ok=True)
CACHE_IMAGE_EMB = CACHE_DIR / 'image_embeddings.pkl'

DISCLAIMER = 'This image is unavailable in your country in compliance with local laws.'

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)


## Шаг 1. Загрузка данных и EDA


In [ ]:

train_df = pd.read_csv(DATA_DIR / 'train_dataset.csv')
expert_df = pd.read_csv(
    DATA_DIR / 'ExpertAnnotations.tsv', sep='\t', header=None,
    names=['image', 'query_id', 'expert_1', 'expert_2', 'expert_3'],
)
crowd_df = pd.read_csv(
    DATA_DIR / 'CrowdAnnotations.tsv', sep='\t', header=None,
    names=['image', 'query_id', 'share_yes', 'count_yes', 'count_no'],
)

print('train_dataset:', train_df.shape)
print('уникальных изображений:', train_df['image'].nunique())
print('уникальных описаний:', train_df['query_id'].nunique())
print('ExpertAnnotations:', expert_df.shape)
print('CrowdAnnotations:', crowd_df.shape)
print('\nПримеры train_dataset:')
display(train_df.head(3))


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ['expert_1', 'expert_2', 'expert_3']):
    expert_df[col].value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title(f'Распределение {col}')
    ax.set_xlabel('Оценка (1–4)')
plt.tight_layout()
plt.show()

print('Крауд share_yes:')
print(crowd_df['share_yes'].describe())

plt.figure(figsize=(8, 4))
sns.histplot(crowd_df['share_yes'], bins=20, kde=True)
plt.title('Доля подтверждений (крауд)')
plt.xlabel('share_yes')
plt.show()


**EDA**

- 5822 пары «изображение — текст», 1000 уникальных изображений.
- Экспертные оценки: шкала 1–4, мода — 1 (несоответствие).
- Крауд: медиана share_yes = 0, большинство пар не подтверждено.
- Датасеты согласованы по ключам image + query_id.


In [ ]:

def aggregate_expert_scores(row):
    votes = [row['expert_1'], row['expert_2'], row['expert_3']]
    rating, votes_count = Counter(votes).most_common(1)[0]
    return rating if votes_count >= 2 else np.nan

expert_df['expert_score'] = expert_df.apply(aggregate_expert_scores, axis=1)
excluded = expert_df['expert_score'].isna().sum()
print(f'Исключено пар (разногласие экспертов): {excluded} ({excluded / len(expert_df):.1%})')

merged_df = (
    train_df
    .merge(expert_df[['image', 'query_id', 'expert_score']], on=['image', 'query_id'], how='left')
    .merge(crowd_df[['image', 'query_id', 'share_yes']], on=['image', 'query_id'], how='left')
)
model_df = merged_df.dropna(subset=['expert_score']).copy()

model_df['expert_target'] = (model_df['expert_score'] - 1) / 3
model_df['target'] = model_df['expert_target']
crowd_mask = model_df['share_yes'].notna()
model_df.loc[crowd_mask, 'target'] = (
    0.6 * model_df.loc[crowd_mask, 'expert_target']
    + 0.4 * model_df.loc[crowd_mask, 'share_yes']
)

print('Строк после агрегации:', len(model_df))
print('target:')
print(model_df['target'].describe())

plt.figure(figsize=(8, 4))
sns.histplot(model_df['target'], bins=20, kde=True)
plt.title('Целевая переменная target ∈ [0, 1]')
plt.show()


**Целевая переменная**

- Агрегация экспертов: голосование большинства (2 из 3); при полном разногласии — исключение.
- Финальный target: 0.6 × expert_target + 0.4 × share_yes (если есть крауд-оценка).
- expert_target = (оценка − 1) / 3 → диапазон [0, 1].
- После фильтрации экспертов: 5696 строк; средний target ≈ 0.17.


## Шаг 2. Юридические ограничения


In [ ]:

CHILD_KEYWORDS = [
    'child', 'children', 'kid', 'kids', 'baby', 'babies', 'toddler', 'infant',
    'teen', 'teenager', 'tyke', 'son', 'daughter', 'boy', 'boys', 'girl', 'girls',
    'young boy', 'young girl', 'young child', 'little boy', 'little girl', 'little blond',
]

CHILD_PATTERN = re.compile(
    r'\b(?:child(?:ren)?|kid(?:s)?|bab(?:y|ies)|toddler|infant|'
    r'young (?:boy|girl|child)|little (?:boy|girl|blond)|teen(?:ager)?|'
    r'tyke|son|daughter|boy(?:s)?|girl(?:s)?)\b',
    re.I,
)

train_df['query_image'] = train_df['query_id'].str.split('#').str[0]
self_descriptions = train_df[train_df['image'] == train_df['query_image']]

banned_images = set(
    self_descriptions.loc[
        self_descriptions['query_text'].str.contains(CHILD_PATTERN, na=False),
        'image',
    ]
)

rows_before = len(model_df)
model_df = model_df[~model_df['image'].isin(banned_images)].copy()

print('Ключевые слова:', CHILD_KEYWORDS[:8], '...')
print('Изображений с детьми в собственных описаниях:', len(banned_images))
print('Удалено строк:', rows_before - len(model_df))
print('Осталось строк:', len(model_df), '| изображений:', model_df['image'].nunique())
if banned_images:
    print('Примеры исключённых файлов:', sorted(banned_images)[:5])


**Фильтрация**

- Список слов: child, kid, baby, toddler, infant, teen, boy, girl и др.
- Критерий: собственное описание изображения (image == query_image) содержит ключевые слова.
- Удалено 204 строки, 37 изображений.
- Осталось 5492 пары, 963 изображения.


## Шаг 3. Векторизация изображений (ResNet50, PyTorch)


In [ ]:

resnet50 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
for param in resnet50.parameters():
    param.requires_grad_(False)

modules = list(resnet50.children())[:-1]
image_encoder = nn.Sequential(*modules)
image_encoder.eval()

image_preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def encode_image(image_path):
    img = Image.open(image_path).convert('RGB')
    tensor = image_preprocess(img).unsqueeze(0)
    with torch.no_grad():
        embedding = image_encoder(tensor).flatten()
    return embedding.numpy()

unique_train_images = model_df['image'].unique()
print('Уникальных изображений для кодирования:', len(unique_train_images))


In [ ]:

if CACHE_IMAGE_EMB.exists():
    with open(CACHE_IMAGE_EMB, 'rb') as f:
        image_embeddings = pickle.load(f)
    print('Загружен кэш эмбеддингов:', len(image_embeddings))
else:
    image_embeddings = {}
    for image_name in tqdm(unique_train_images, desc='ResNet50'):
        image_embeddings[image_name] = encode_image(TRAIN_IMAGES_DIR / image_name)
    with open(CACHE_IMAGE_EMB, 'wb') as f:
        pickle.dump(image_embeddings, f)
    print('Кэш сохранён')

sample_name = unique_train_images[0]
sample_img_emb = image_embeddings[sample_name]
print('Размерность эмбеддинга изображения:', sample_img_emb.shape)
print('Первые 5 значений:', sample_img_emb[:5])


**Эмбеддинги изображений**

- Модель: ResNet50 (ImageNet), без FC-слоёв, Global Average Pooling.
- Размерность вектора: **2048**.
- Веса заморожены, режим eval().


## Шаг 3. Векторизация текстов (TF-IDF)


In [ ]:

unique_texts = model_df['query_text'].unique()
print('Уникальных текстов:', len(unique_texts))

tfidf_vectorizer = TfidfVectorizer(
    max_features=500,
    ngram_range=(1, 2),
    stop_words='english',
    min_df=2,
)
tfidf_vectorizer.fit(unique_texts)

text_embeddings = {
    text: tfidf_vectorizer.transform([text]).toarray()[0]
    for text in unique_texts
}

sample_text = unique_texts[0]
sample_text_emb = text_embeddings[sample_text]
print('Размерность TF-IDF-вектора:', sample_text_emb.shape)
print('Пример текста:', sample_text[:90], '...')
print('Топ-5 признаков:', list(tfidf_vectorizer.get_feature_names_out()[sample_text_emb.argsort()[-5:][::-1]]))


**Эмбеддинги текстов**

- Метод: TF-IDF, uni- и биграммы, max_features=500, stop_words='english'.
- Размерность вектора: **500**.
- Конкатенация [image_emb ∥ text_emb]: **2548** признаков на пару.


## Подготовка матрицы признаков


In [ ]:

feature_rows = []
for row in model_df.itertuples(index=False):
    combined = np.concatenate([
        image_embeddings[row.image],
        text_embeddings[row.query_text],
    ])
    feature_rows.append(combined)

X = np.vstack(feature_rows)
y = model_df['target'].values
groups = model_df['image'].values

print('X:', X.shape)
print('y:', y.shape)
print('Групп (изображений):', len(np.unique(groups)))
print('Средний target:', round(y.mean(), 3))


## Шаг 3. Обучение моделей


In [ ]:

gss = GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=RANDOM_STATE)
train_idx, val_idx = next(gss.split(X, y, groups=groups))

X_train, X_val = X[train_idx], X[val_idx]
y_train, y_val = y[train_idx], y[val_idx]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print('Train:', X_train.shape[0], '| Val:', X_val.shape[0])
print('Train images:', len(np.unique(groups[train_idx])))
print('Val images:', len(np.unique(groups[val_idx])))


In [ ]:

models_config = {
    'LinearRegression': LinearRegression(),
    'Ridge (alpha=1)': Ridge(alpha=1.0, random_state=RANDOM_STATE),
    'Ridge (alpha=10)': Ridge(alpha=10.0, random_state=RANDOM_STATE),
    'MLP (128)': MLPRegressor(hidden_layer_sizes=(128,), max_iter=400, random_state=RANDOM_STATE, early_stopping=True),
    'MLP (256,128)': MLPRegressor(hidden_layer_sizes=(256, 128), max_iter=400, random_state=RANDOM_STATE, early_stopping=True),
    'MLP (512,256,128)': MLPRegressor(
        hidden_layer_sizes=(512, 256, 128), max_iter=400,
        random_state=RANDOM_STATE, early_stopping=True,
    ),
}

results = []
trained_models = {}

for name, model in models_config.items():
    model.fit(X_train_scaled, y_train)
    y_pred = np.clip(model.predict(X_val_scaled), 0, 1)
    mae = mean_absolute_error(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    r2 = r2_score(y_val, y_pred)
    auc = roc_auc_score((y_val >= 0.5).astype(int), y_pred)
    results.append({'model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2, 'ROC-AUC': auc})
    trained_models[name] = model

results_df = pd.DataFrame(results).sort_values('MAE')
display(results_df.round(4))

best_name = results_df.iloc[0]['model']
best_model = trained_models[best_name]
print(f'\nЛучшая модель по MAE: {best_name}')


In [ ]:

plt.figure(figsize=(8, 4))
sns.barplot(data=results_df, x='MAE', y='model', hue='model', legend=False, palette='Blues_r')
plt.title('Сравнение моделей (MAE ↓ лучше)')
plt.xlabel('MAE')
plt.tight_layout()
plt.show()

y_pred_best = np.clip(best_model.predict(X_val_scaled), 0, 1)
plt.figure(figsize=(6, 6))
plt.scatter(y_val, y_pred_best, alpha=0.35, s=15)
plt.plot([0, 1], [0, 1], 'r--', lw=1)
plt.xlabel('Истинный target')
plt.ylabel('Предсказание')
plt.title(f'Предсказания: {best_name}')
plt.show()


**Обучение**

- Разбиение: GroupShuffleSplit 70/30 по image (без утечки между train/val).
- Метрика: **MAE** — интерпретируемая ошибка на шкале [0, 1].
- Модели: линейная регрессия, Ridge, MLP (3 конфигурации).
- Лучшая: **MLP (512,256,128)** — MAE ≈ 0.19, ROC-AUC ≈ 0.70.
- Линейные модели слабее; простой MLP (128) переобучается.


In [ ]:

X_scaled_full = scaler.fit_transform(X)
best_model.fit(X_scaled_full, y)
print('Модель дообучена на полной выборке.')


## Шаг 4. Тестирование и демо-поиск


In [ ]:

test_queries = pd.read_csv(DATA_DIR / 'test_queries.csv', sep='|', skiprows=[1])
test_image_list = pd.read_csv(DATA_DIR / 'test_images.csv')['image'].tolist()

print('Тестовых запросов:', len(test_queries))
print('Тестовых изображений:', len(test_image_list))

test_image_embeddings = {}
for image_name in tqdm(test_image_list, desc='Test images'):
    test_image_embeddings[image_name] = encode_image(TEST_IMAGES_DIR / image_name)


In [ ]:

def vectorize_text(text):
    return tfidf_vectorizer.transform([text]).toarray()[0]

def predict_similarity(query_text, image_name, image_embeddings_dict):
    text_vec = vectorize_text(query_text)
    img_vec = image_embeddings_dict[image_name]
    features = scaler.transform(np.concatenate([img_vec, text_vec]).reshape(1, -1))
    return float(np.clip(best_model.predict(features)[0], 0, 1))

def search_image(query_text, candidate_images, image_embeddings_dict):
    if CHILD_PATTERN.search(query_text):
        return DISCLAIMER, None, None

    scores = {
        image_name: predict_similarity(query_text, image_name, image_embeddings_dict)
        for image_name in candidate_images
    }
    best_image = max(scores, key=scores.get)
    return best_image, scores[best_image], scores

harmful_query = 'A baby is playing in the park with toys.'
result = search_image(harmful_query, test_image_list, test_image_embeddings)
print('Запрос с детьми:', harmful_query)
print('Результат:', result[0])


In [ ]:

sample_queries = test_queries.sample(10, random_state=RANDOM_STATE).reset_index(drop=True)

fig, axes = plt.subplots(10, 2, figsize=(10, 28))
hits = []

for i, row in sample_queries.iterrows():
    best_image, best_score, _ = search_image(
        row['query_text'], test_image_list, test_image_embeddings,
    )
    gt_image = row['image']

    axes[i, 0].imshow(Image.open(TEST_IMAGES_DIR / gt_image))
    axes[i, 0].set_title('Эталон', fontsize=9)
    axes[i, 0].axis('off')

    if best_image == DISCLAIMER:
        axes[i, 1].text(0.5, 0.5, DISCLAIMER, ha='center', va='center', wrap=True, fontsize=8)
        axes[i, 1].set_title('Дисклеймер', fontsize=9)
        hits.append(False)
    else:
        hit = best_image == gt_image
        hits.append(hit)
        axes[i, 1].imshow(Image.open(TEST_IMAGES_DIR / best_image))
        marker = '✓' if hit else '✗'
        axes[i, 1].set_title(f'Модель {best_score:.2f} {marker}', fontsize=9)
    axes[i, 1].axis('off')
    axes[i, 0].set_ylabel(row['query_text'][:55] + '...', fontsize=7, rotation=0, labelpad=90, va='center')

plt.suptitle('Эталон vs найденное изображение (10 запросов)', y=1.01)
plt.tight_layout()
plt.show()

print(f'Top-1 accuracy: {np.mean(hits):.0%} ({sum(hits)}/10)')


In [ ]:

manual_queries = [
    'A brown dog sits in long grass.',
    'Two men are playing soccer on a field.',
    'A hiker poses for a picture in front of stunning mountains.',
    'A toddler smiles at the camera.',
]

for q in manual_queries:
    result, score, _ = search_image(q, test_image_list, test_image_embeddings)
    if result == DISCLAIMER:
        print(f'[DISCLAIMER] {q}')
    else:
        print(f'[{score:.2f}] {q[:60]} -> {result}')


**Тестирование**

- Top-1 accuracy на 10 случайных запросах: 0/10.
- Типичные ошибки: путаница сцен с людьми/животными, игнорирование деталей запроса.
- Модель склонна выбирать «универсальные» изображения с высоким откликом.
- Дисклеймер срабатывает при ключевых словах о детях в запросе.
- PoC подтверждает техническую реализуемость; для продакшена нужны CLIP/BERT и больший корпус.


## Шаг 5. Выводы

**Лучшая модель:** MLP (512, 256, 128) + ResNet50 + TF-IDF.

**Размерности:** изображение 2048, текст 500, конкатенация 2548.

**Метрика:** MAE на val ≈ 0.19.

**Ошибки поиска:**
- неверная семантика (собака → другое животное);
- потеря контекста (несколько людей/объектов);
- слабая генерализация на 100 тестовых изображений.

**Практическая осуществимость:** PoC работает end-to-end. Для MVP — multimodal embeddings (CLIP), ANN-индекс, фильтрация контента, A/B-тесты с фотографами.
